In [21]:
import openai, json

client = openai.OpenAI()
messages = []

In [22]:
# import urllib.request
import requests, json

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


def get_popular_movies():
    responses = requests.get(f"{BASE_URL}/movies")
    return json.dumps(responses.json())

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return json.dumps(response.json())

def get_movie_similar(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return json.dumps(response.json())

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_movie_similar": get_movie_similar,
}

# print(get_popular_movies())

In [23]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 조회합니다. 인기 영화 목록을 요청할 때만 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 상세 정보(줄거리, 장르, 개봉일, 평점 등)를 조회합니다. '더 알려줘', '상세 정보' 요청에 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "영화 ID"},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화의 출연진 및 제작진을 조회합니다. 사용자가 출연진·배우·제작진을 명시적으로 물을 때만 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "영화 ID"},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "비슷한 영화 목록을 조회합니다. 사용자가 유사 영화·추천 영화를 명시적으로 물을 때만 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "영화 ID"},
                },
                "required": ["id"],
            },
        },
    },
]

In [24]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"Calling function: {function_name} with {arguments}")
            print(f"Agent: {function_name}() 호출")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        
        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai() 

User: 지금 인기 있는 영화 알려줘
Agent: get_popular_movies() 호출
Agent: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - 개봉일: 2026년 5월 13일
   - 줄거리: "One Wish Willow"를 깨뜨린 후, 열망하는 사랑을 얻은 로맨틱 남자가 그 원하는 바가 어둡고 사악한 대가를 치르게 되는 이야기를 담고 있습니다.
   - 평점: 7.9
   - ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - 개봉일: 2026년 6월 3일
   - 줄거리: 1980년대 안드라 프라데시의 시골 마을에서 한 열정적인 마을 사람이 스포츠를 통해 자신의 커뮤니티를 단결시키는 이야기입니다.
   - 평점: 6.47
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - 개봉일: 2026년 6월 4일
   - 줄거리: 결혼을 포기한 Jass가 새로운 사랑을 찾지만, 충격적인 사실로 인해 사랑, 충성심, 그리고 헌신의 진정한 의미에 직면하는 이야기입니다.
   - 평점: 5.5
   - ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**
   - 개봉일: 2021년 10월 16일
   - 줄거리: 벨기에 작가인 루이의 상상력을 불러일으키기 위해 프랑스 리비에라에서 고립된 생활을 하게 되는 이야기를 다룹니다.
   - 평점: 8.2
   - ![The Unknown Man](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaY